# Jarvis Voice — Kaggle B: Cloned OmniVoice TTS

Dedicated GPU for English and Tamil cloning. **No Whisper or Groq API key here.**

Enable **GPU T4** and Internet. Attach your private voice dataset containing `jarvis_en/new-ref.mp3`, `jarvis_en/ref.txt`, `jarvis_ta/ref.mp3`, `jarvis_ta/ref.txt`. Your other Kaggle account needs legitimate access to this private dataset.

Create Kaggle Secret `TTS_WS_TOKEN` (different from `STT_WS_TOKEN`, 24+ random characters). Run cells top-to-bottom.

## 1. Pull the latest repository

In [ ]:
import os, sys, time, subprocess, site, re
from pathlib import Path
REPO=Path("/kaggle/working/jarvis-realtime-voice")
if (REPO/".git").exists():
    subprocess.run(["git","-C",str(REPO),"pull","--ff-only"],check=True)
else:
    subprocess.run(["git","clone","https://github.com/Godf-Cyb3r-1/jarvis-realtime-voice.git",str(REPO)],check=True)
os.chdir(REPO)
print("Repo:",REPO)

## 2. Install TTS packages if missing (restart session after installation)

In [ ]:
import importlib.util
needed={"omnivoice":"omnivoice","fastapi":"fastapi","uvicorn":"uvicorn",
        "numpy":"numpy","soundfile":"soundfile","scipy":"scipy","torch":"torch"}
missing=[name for name,mod in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:",missing)
    subprocess.run([sys.executable,"-m","pip","install","-r",
                    str(REPO/"requirements-realtime-tts.txt")],check=True)
    if "omnivoice" in missing:
        subprocess.run([sys.executable,"-m","pip","install","omnivoice"],check=True)
    raise RuntimeError("Restart Python session and rerun all cells")
import torch
assert torch.cuda.is_available(),"Enable GPU acceleration"
print("GPU:",torch.cuda.get_device_name(0))

## 3. Prepare PRIVATE voice references exactly once

In [ ]:
import shutil, soundfile as sf
matches=[]
for mp3 in Path("/kaggle/input").rglob("new-ref.mp3"):
    d=mp3.parent.parent
    if (d/"jarvis_en/ref.txt").is_file() and (d/"jarvis_ta/ref.mp3").is_file() and (d/"jarvis_ta/ref.txt").is_file():
        matches.append(d)
assert matches,"Voice dataset not attached or not shared with this Kaggle account"
DATASET=matches[0]
VOICE_DIR=REPO/"voices"
VOICE_DIR.mkdir(exist_ok=True)
for lang,rel,audio in [
    ("en","jarvis_en/new-ref.mp3","jarvis_en/ref.txt"),
    ("ta","jarvis_ta/ref.mp3","jarvis_ta/ref.txt")
]:
    shutil.copy2(DATASET/audio,VOICE_DIR/f"{lang}.txt")
    subprocess.run(["ffmpeg","-y","-v","error","-i",str(DATASET/rel),
        "-ar","24000","-ac","1",str(VOICE_DIR/f"{lang}.wav")],check=True)
    assert sf.info(VOICE_DIR/f"{lang}.wav").frames>0
print("English/Tamil private reference voices ready")

## 4. Voice GPU configuration, private token and CUDA library paths

In [ ]:
from kaggle_secrets import UserSecretsClient
key=UserSecretsClient().get_secret("TTS_WS_TOKEN")
assert key and len(key)>=24,"Add TTS_WS_TOKEN in Kaggle Secrets"
os.environ["TTS_WS_TOKEN"]=key
del key
os.environ.update({
    "TTS_PROVIDER":"omnivoice",
    "TTS_GPU":"0",  # GPU0 on this independent Kaggle session
    "VOICE_DIR":str(VOICE_DIR),
    "OMNIVOICE_MODEL":"k2-fsa/OmniVoice",
    "OMNIVOICE_STEPS":"12",
    "OMNIVOICE_SPEED":"1.0",
    "TTS_PREFILL_CACHE_LIMIT":"8",
    "PYTHONUNBUFFERED":"1"
})
libs=[]
for sd in site.getsitepackages():
    for name in ("cublas","cudnn","cuda_runtime"):
        p=Path(sd)/"nvidia"/name/"lib"
        if p.is_dir():libs.append(str(p))
old=os.environ.get("LD_LIBRARY_PATH","")
os.environ["LD_LIBRARY_PATH"]=":".join(libs+([old] if old else []))
print("TTS configured, key remains private")

## 5. Start one cloned-voice server and verify health

In [ ]:
subprocess.run([sys.executable,"-m","compileall","-q",
    "realtime_tts_server.py","tts_engine.py","tts_longform.py"],check=True)
import requests
if "server" in globals() and server and server.poll() is None:
    server.terminate();server.wait(timeout=20)
SERVER_LOG=Path("/kaggle/working/jarvis_tts_server.log")
server_handle=SERVER_LOG.open("w")
server=subprocess.Popen([sys.executable,"-m","uvicorn","realtime_tts_server:app",
    "--host","127.0.0.1","--port","8092"],cwd=REPO,env=os.environ.copy(),
    stdout=server_handle,stderr=subprocess.STDOUT)
health=None
for i in range(90):
    if server.poll() is not None:break
    try:
        x=requests.get("http://127.0.0.1:8092/health",timeout=2)
        if x.ok:health=x.json();break
    except requests.RequestException:pass
    time.sleep(1)
if not health:
    print(SERVER_LOG.read_text(errors="replace")[-4000:])
    raise RuntimeError("TTS server failed to start")
print("TTS health:",health)

## 6. Test English AND Tamil in the exact server process before tunneling

In [ ]:
from IPython.display import display,Audio
HEADERS={"Authorization":"Bearer "+os.environ["TTS_WS_TOKEN"]}
for lang,text in [("en","Hey! It's good to hear from you."),
                  ("ta","வணக்கம், எப்படி இருக்கீங்க?")]:
    t0=time.monotonic()
    r=requests.post("http://127.0.0.1:8092/debug/tts",
        headers=HEADERS,json={"text":text,"language":lang},timeout=240)
    if not r.ok:
        print(SERVER_LOG.read_text(errors="replace")[-3500:])
        raise RuntimeError(f"Voice test failed ({lang}): {r.status_code} {r.text[:200]}")
    p=Path("/kaggle/working")/f"jarvis_voice_probe_{lang}.wav"
    p.write_bytes(r.content)
    print(lang,"audio seconds:",round(sf.info(p).duration,2),
          "generation seconds:",round(time.monotonic()-t0,2))
    display(Audio(filename=str(p)))
print("Voice tests passed. Listen before starting the tunnel.")

## 7. Create TTS tunnel and copy TTS_WS_URL to Windows .env

In [ ]:
import re, requests
CLOUDFLARED=Path("/kaggle/working/cloudflared")
if not CLOUDFLARED.is_file():
    subprocess.run(["wget","-q","https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64","-O",str(CLOUDFLARED)],check=True)
CLOUDFLARED.chmod(0o755)
if "tunnel" in globals() and tunnel and tunnel.poll() is None:
    tunnel.terminate()
    tunnel.wait(timeout=15)
TUNNEL_LOG=Path("/kaggle/working/tts_tunnel.log")
tunnel_handle=TUNNEL_LOG.open("w")
tunnel=subprocess.Popen([str(CLOUDFLARED),"tunnel","--url","http://127.0.0.1:8092"],
    stdout=tunnel_handle,stderr=subprocess.STDOUT)
url=None
for _ in range(70):
    if tunnel.poll() is not None:break
    time.sleep(1)
    matches=re.findall(r"https://[a-z0-9-]+\\.trycloudflare\\.com",
                     TUNNEL_LOG.read_text(errors="replace"))
    if matches:
        url=matches[-1].replace("https://","wss://")+"/tts"
        break
if not url:
    print(TUNNEL_LOG.read_text(errors="replace")[-2000:])
    raise RuntimeError("Tunnel failed. Check Internet and cloudflared logs")
print("TTS_WS_URL=",url)
print("Keep the Kaggle secret private; only copy this WSS URL into Windows .env")

## 8. Inspect TTS logs if needed

In [ ]:
print(SERVER_LOG.read_text(errors="replace")[-3000:])
print(TUNNEL_LOG.read_text(errors="replace")[-500:])